In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
# for dirname, _, filenames in os.walk('/kaggle/input'):
#     for filename in filenames:
#         print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [2]:
data = pd.read_csv("/kaggle/input/jan-2026-dl-gen-ai-project/messy_mashup/sample_submission.csv")
data.to_csv("submission.csv",index=False)

In [3]:
import os
import glob
import numpy as np
import pandas as pd
from tqdm import tqdm
import librosa
import librosa.display
import matplotlib.pyplot as plt
import random
import torch

import warnings
warnings.filterwarnings("ignore")

In [4]:
#----------------------------- DON'T CHANGE THIS --------------------------
DATA_SEED = 67
TRAINING_SEED = 1234
SR = 22050
DURATION = 5.0
N_FFT = 2048
HOP_LENGTH = 512
N_MELS = 128
TOP_DB=20
TARGET_SNR_DB = 10

random.seed(DATA_SEED)
np.random.seed(DATA_SEED)
torch.manual_seed(DATA_SEED)
torch.cuda.manual_seed(DATA_SEED)

In [5]:
# CONFIGURATION
DATA_ROOT = "/kaggle/input/jan-2026-dl-gen-ai-project/messy_mashup"
GENRES = [
    'blues', 'classical', 'country', 'disco', 'hiphop', 
    'jazz', 'metal', 'pop', 'reggae', 'rock'
]
STEMS = {
    'drums.wav': 'drums',
    'vocals.wav': 'vocals',
    'bass.wav': 'bass',
    'other.wav': 'other'
}
STEM_KEYS = ['drums', 'vocals', 'bass', 'other']
GENRE_TO_TEST = 'rock'
SONG_INDEX = 0

In [6]:
def build_dataset(root_dir, val_split=0.17, seed=42):
    train_dataset = {g: {s.replace('.wav', ''): [] for s in STEM_KEYS} for g in GENRES}
    val_dataset   = {g: {s.replace('.wav', ''): [] for s in STEM_KEYS} for g in GENRES}

    rng = random.Random(seed)
    
    # Tracking variables for Q1 & Q2
    corrupted_count = 0
    less_than_5_0491MB = 0
    greater_than_5_0493MB = 0
    
    # 1MB = 1024 * 1024 bytes
    threshold_low = 5.0491 * 1024 * 1024
    threshold_high = 5.0493 * 1024 * 1024

    for genre in GENRES:
        genre_path = os.path.join(root_dir, 'genres_stems', genre)
        if not os.path.exists(genre_path): continue
        
        # Get list of song folders
        songs = sorted([d for d in os.listdir(genre_path) if os.path.isdir(os.path.join(genre_path, d))])
        valid_songs = []

        for song in songs:
            song_path = os.path.join(genre_path, song)
            stems_present = True
            
            for s_key in STEM_KEYS:
                f_path = os.path.join(song_path, f"{s_key}.wav")
                if not os.path.exists(f_path):
                    stems_present = False
                    break
                
                f_size = os.path.getsize(f_path)
                if f_size < 4096: # 4kb
                    corrupted_count += 1
                    stems_present = False
                
                if f_size < threshold_low: less_than_5_0491MB += 1
                if f_size > threshold_high: greater_than_5_0493MB += 1
            
            if stems_present:
                valid_songs.append(song)

        # Shuffle and Split
        rng.shuffle(valid_songs)
        split_point = int(len(valid_songs) * (1 - val_split))
        tr_songs, val_songs = valid_songs[:split_point], valid_songs[split_point:]

        for s in tr_songs:
            for k in STEM_KEYS:
                train_dataset[genre][k].append(os.path.join(genre_path, s, f"{k}.wav"))
        for s in val_songs:
            for k in STEM_KEYS:
                val_dataset[genre][k].append(os.path.join(genre_path, s, f"{k}.wav"))

    print(f"Q1 Answer: {corrupted_count + less_than_5_0491MB}")
    print(f"Q2 Answer: {abs(greater_than_5_0493MB - less_than_5_0491MB)}")
    return train_dataset, val_dataset

tr, val = build_dataset(DATA_ROOT)
print(f"Q3 Answer: {abs(len(tr['reggae']['drums']) - len(val['country']['vocals']))}")

Q1 Answer: 1256
Q2 Answer: 1072
Q3 Answer: 66


In [7]:
def find_long_silences(dataset_dict, sr=SR, threshold_sec=DURATION, top_db=TOP_DB):
    records = []
    for genre, stems in tqdm(dataset_dict.items()):
        for stem_name, file_paths in stems.items():
            for path in file_paths:
                y, _ = librosa.load(path, sr=sr)
                duration = librosa.get_duration(y=y, sr=sr)
                intervals = librosa.effects.split(y, top_db=top_db)
                
                max_silence = 0
                locs = []
                
                if len(intervals) == 0:
                    max_silence = duration
                    locs = ["full"]
                else:
                    # Check Start/End/Middle
                    start_gap = intervals[0][0] / sr
                    end_gap = (len(y) - intervals[-1][1]) / sr
                    
                    if start_gap >= threshold_sec: locs.append("start")
                    if end_gap >= threshold_sec: locs.append("end")
                    
                    max_silence = max(start_gap, end_gap)
                    
                    for i in range(len(intervals)-1):
                        mid_gap = (intervals[i+1][0] - intervals[i][1]) / sr
                        if mid_gap >= threshold_sec:
                            if "middle" not in locs: locs.append("middle")
                        max_silence = max(max_silence, mid_gap)

                if max_silence >= threshold_sec:
                    records.append({
                        "Genre": genre, "Stem": stem_name, "Max_Silence_Sec": max_silence,
                        "Silence_Location": " ".join(locs)
                    })
    return pd.DataFrame(records)

In [8]:
df_silence = find_long_silences(tr, threshold_sec=5.0, top_db=TOP_DB)

q4_ans = len(df_silence)

vocals_df = df_silence[df_silence['Stem'] == 'vocals']
q5_ans = len(vocals_df)

q6_ans = vocals_df['Max_Silence_Sec'].mean() if not vocals_df.empty else 0

jazz_drums = df_silence[(df_silence['Genre'] == 'jazz') & (df_silence['Stem'] == 'drums')]
q7_ans = len(jazz_drums)

jazz_drums_mid_only = jazz_drums[jazz_drums['Silence_Location'].str.strip() == 'middle']
q8_ans = len(jazz_drums_mid_only)

jazz_drums_10s = jazz_drums[jazz_drums['Max_Silence_Sec'] >= 10]
q9_ans = len(jazz_drums_10s)

print(f"Answer Q4: {q4_ans}")
print(f"Answer Q5: {q5_ans}")
print(f"Answer Q6: {round(q6_ans, 2)}")
print(f"Answer Q7: {q7_ans}")
print(f"Answer Q8: {q8_ans}")
print(f"Answer Q9: {q9_ans}")

100%|██████████| 10/10 [06:11<00:00, 37.15s/it]

Answer Q4: 680
Answer Q5: 304
Answer Q6: 12.59
Answer Q7: 24
Answer Q8: 13
Answer Q9: 7


In [9]:
# Select first rock song from 'tr'
rock_song_paths = [tr['rock'][k][SONG_INDEX] for k in STEM_KEYS]
stems_audio = [librosa.load(p, sr=SR, duration=DURATION)[0] for p in rock_song_paths]

stems_stack = np.array(stems_audio)
mix_raw = np.sum(stems_stack, axis=0)

rms_val = np.sqrt(np.mean(mix_raw**2))
max_val = np.max(np.abs(mix_raw))
mix_norm = mix_raw / max_val

print(f"Q10: {len(mix_raw)}")
print(f"Q11: {round(rms_val, 2)}")
print(f"Q12: {round(max_val, 2)}")

Q10: 110250
Q11: 0.10000000149011612
Q12: 0.5899999737739563
